# 05 -- Potencjal produkcji energii

Ten notebook obejmuje:

1. **Parametry instalowane** -- gestosc, przyspieszenie, spad, sprawnosci
2. **Model spadu** -- zaleznosc spadu netto od przeplywu (H = f(Q))
3. **Punkt projektowy** -- wybor dnia instalacyjnego na krzywej uporzadkowanej
4. **Obliczenie mocy i energii** -- P(t) = rho * g * Qwpp(t) * H(t) * eta
5. **Ograniczenia eksploatacyjne** -- Qmin, Hmin
6. **Optymalizacja punktu projektowego** -- przeglad dni instalacyjnych
7. **Analiza ekonomiczna** -- koszty, przychod, NPV / LCOE / okres zwrotu

Dane wejsciowe: dane oczyszczone z nb 02; sredni rok uporzadkowany Q budujemy tutaj z interpolacji do lokalizacji EW (nie z nb 04, ktory liczy go dla Malczyc).

---

## Konfiguracja

### Modul `src/watershed.py` -- nowy modul

**Prompt do LLM tworzacy ten modul:**
> *"Stworz modul src/watershed.py do obslugi zlewnii:
> 1. `load_gauges()` -- wczytaj dane wodowskazow IMGW z data/imgw_gauges.json
> 2. `find_gauge(station_id)` -- znajdz wodowskaz po ID
> 3. `get_catchment_area(lat, lng)` -- pobierz powierzchnie zlewni z cache
>    (data/catchment_areas.json) lub oblicz uzywajac delineatora MERIT
> 4. `get_station_area(station_id)` -- pobierz powierzchnie zlewni dla wodowskazu
> 5. `get_catchment_area(lat, lng)` -- powierzchnia zlewni (MERIT, z cache)
>
> Cache (data/catchment_areas.json) pozwala na prace offline -- wystarczy raz
> obliczyc powierzchnie i zapisac wyniki."*

### Modul `src/hydrology.py` -- istniejace funkcje

**Uzyte funkcje:** average_sorted_year, flow_duration_curve, characteristic_flows, interpolate_q_to_location.

*(Filtrowanie lat zrobione juz w nb 02 — `daily_hydro_clean.parquet` jest gotowy do uzycia.)*

**Prompt do LLM:**
> *"Napisz kod konfiguracji notebooka: zaladuj modul src/imgw_data (load_processed) i src/hydrology
> (average_sorted_year, flow_duration_curve, characteristic_flows, interpolate_q_to_location)."*

**Uzyte moduly/funkcje:** `src.imgw_data`: load_processed; `src.hydrology`: average_sorted_year, flow_duration_curve, characteristic_flows, interpolate_q_to_location.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.imgw_data import load_processed
from src.hydrology import (
    average_sorted_year,
    flow_duration_curve,
    characteristic_flows,
    interpolate_q_to_location,
)
from src.watershed import get_station_area, find_gauge, get_catchment_area

pd.set_option('display.max_columns', 15)
print('Moduly zaladowane.')

Moduly zaladowane.


**Prompt do LLM:**
> *"Napisz kod ktory: (1) wczyta dane hydrologiczne, (2) zidentyfikuje stacje
> upstream i downstream na Odrze (Brzeg Dolny = upstream, Malczyce = downstream),
> (3) pobierze powierzchnie zlewni z modulu watershed (z cache lub delineatora),
> (4) obliczy powierzchnie zlewni dla lokalizacji EW (punkt miedzy stacjami).
> Wyswietl informacje o stacjach i zlewniach."*

**Uzyte moduly/funkcje:** `src.imgw_data.load_processed()`, `src.watershed.get_station_area()`, `src.watershed.find_gauge()`, `src.watershed.get_catchment_area()`

In [2]:
# Wczytaj dane (oczyszczone w notebooku 02)
df = load_processed('../data/processed/daily_hydro_clean.parquet')

# Stacje na Odrze (Brzeg Dolny jest upstream, Malczyce downstream)
STATION_UP = '151160170'    # Brzeg Dolny (upstream, mniejsza zlewnia)
STATION_DOWN = '151160150'  # Malczyce (downstream, wieksza zlewnia)

# Powierzchnie zlewni [km2] -- z cache (obliczone delineatorem MERIT)
A_UP = get_station_area(STATION_UP)
A_DOWN = get_station_area(STATION_DOWN)

# Lokalizacja EW -- punkt miedzy wodowskazami
gauge_up = find_gauge(STATION_UP)
gauge_down = find_gauge(STATION_DOWN)
lat_ew = (gauge_up['lat'] + gauge_down['lat']) / 2
lng_ew = (gauge_up['lng'] + gauge_down['lng']) / 2
A_TARGET = get_catchment_area(lat_ew, lng_ew, label='EW')

for sid, label in [(STATION_UP, 'upstream'), (STATION_DOWN, 'downstream')]:
    g = find_gauge(sid)
    row = df[df['station_id'] == sid].iloc[0]
    n = len(df[df['station_id'] == sid])
    area = get_station_area(sid)
    print(f'{g["name"]} ({label}): {row["station_name"]} na {row["river_name"]}, A={area:.0f} km2, {n} dni')

print(f'\nLokalizacja EW: ({lat_ew:.4f}, {lng_ew:.4f}), A = {A_TARGET:.0f} km2')
print(f'  (miedzy {gauge_up["name"]} a {gauge_down["name"]})')

Brzeg Dolny (upstream): BRZEG DOLNY na Odra (1), A=26495 km2, 14975 dni
Malczyce (downstream): MALCZYCE na Odra (1), A=26860 km2, 14975 dni

Lokalizacja EW: (51.2433, 16.6097), A = 26545 km2
  (miedzy Brzeg Dolny a Malczyce)


---
## Krok 1: Powierzchnie zlewni i interpolacja przeplywu

### 1a. Powierzchnie zlewni

Powierzchnie zlewni sa potrzebne do interpolacji przeplywu miedzy wodowskazami.
Obliczane sa delineatorem MERIT (na bazie globalnych danych hydrologicznych)
i zapisywane w cache (`data/catchment_areas.json`).

**Jesli cache jest pusty:**
- modul `src/watershed.py` sprobuje uruchomic delineator automatycznie
- alternatywnie: dodaj wpis recznie do `data/catchment_areas.json`

### 1b. Interpolacja przeplywu

Lokalizacja EW lezy miedzy dwoma wodowskazami (Brzeg Dolny i Malczyce).
Uzywamy **metody interpolacyjnej** -- interpolacja potegowa wazona powierzchnia zlewni:

$$n(t) = \frac{\ln Q_{down}(t) - \ln Q_{up}(t)}{\ln A_{down} - \ln A_{up}}$$

$$Q_{EW}(t) = Q_{up}(t) \cdot \left(\frac{A_{EW}}{A_{up}}\right)^{n(t)}$$

Wykladnik $n$ obliczany jest dla kazdego dnia z przeplywow na obu wodowskazach.

**Prompt do LLM:**
> *"Napisz kod ktory: (1) polaczy dane z dwoch stacji (Malczyce i Brzeg Dolny) po dacie,
> (2) dla kazdego dnia obliczy wykladnik n = ln(Q_down/Q_up) / ln(A_down/A_up),
> (3) obliczy przeplyw w lokalizacji EW: Q_ew = Q_up * (A_ew/A_up)^n,
> (4) utworzy DataFrame z kolumnami date, Q_up, Q_down, Q_ew. Pokaz statystyki."*

**Uzyte moduly/funkcje:** interpolacja potegowa wazona powierzchnia zlewni (wektoryzowana)

In [3]:
# Przygotuj dane z obu stacji (rowniez z kolumna stanu wody do modelu spadu w Krok 4)
df_up = df[df['station_id'] == STATION_UP][['date', 'discharge_m3s', 'water_level_cm']].rename(
    columns={'discharge_m3s': 'Q_up', 'water_level_cm': 'level_up_cm'})
df_down = df[df['station_id'] == STATION_DOWN][['date', 'discharge_m3s', 'water_level_cm']].rename(
    columns={'discharge_m3s': 'Q_down', 'water_level_cm': 'level_down_cm'})

# Polacz po dacie + odrzuc dni z brakami / niedodatnimi Q (dla per-day n metody)
df_ew = df_up.merge(df_down, on='date', how='inner').dropna()
df_ew = df_ew[(df_ew['Q_up'] > 0) & (df_ew['Q_down'] > 0)].copy()

# Interpolacja Q -> lokalizacja EW: wspolna funkcja z src.hydrology
# (te sama uzyta w nb 03 i nb 10 — jeden punkt definicji = jeden punkt zmiany)
df_ew['Q_ew'] = interpolate_q_to_location(
    Q_up=df_ew['Q_up'].values, Q_down=df_ew['Q_down'].values,
    A_up=A_UP, A_down=A_DOWN, A_target=A_TARGET,
    method='daily_n',  # forma WPE_2 (per-day exponent); patrz porownanie ponizej
)

# Statystyki dziennego wykladnika n (dla sygnalu o stabilnosci — szczegoly ponizej)
ln_area_ratio = np.log(A_DOWN / A_UP)
n_exp = np.log(df_ew['Q_down'] / df_ew['Q_up']) / ln_area_ratio

print(f'Liczba dni z danymi z obu stacji: {len(df_ew)}')
print(f'\nStatystyki wykladnika n: mean={n_exp.mean():.2f}, std={n_exp.std():.2f}')
print(f'\nStatystyki przeplywu [m3/s]:')
print(df_ew[['Q_up', 'Q_down', 'Q_ew']].describe().round(2))
print(f'\nPrzykladowe wiersze:')
df_ew.head()

Liczba dni z danymi z obu stacji: 14975

Statystyki wykladnika n: mean=9.02, std=34.20

Statystyki przeplywu [m3/s]:
           Q_up    Q_down      Q_ew
count  14975.00  14975.00  14975.00
mean     131.19    137.36    130.77
std       90.64     94.34     88.92
min        7.92     22.80      9.64
25%       65.60     78.20     68.02
50%      108.58    114.00    109.32
75%      176.88    161.00    174.67
max      764.32   1420.00    812.07

Przykladowe wiersze:


,date,Q_up,level_up_cm,Q_down,level_down_cm,Q_ew
0,1980-11-01,137.320055,202.0,136.0,110.0,137.138609
1,1980-11-02,138.999634,204.0,136.0,110.0,138.585161
2,1980-11-03,138.999634,204.0,138.0,112.0,138.862375
3,1980-11-04,140.683069,206.0,138.0,112.0,140.312745
4,1980-11-05,140.683069,206.0,138.0,112.0,140.312745


In [4]:
# Porownanie trzech metod interpolacji Q -> EW (wspolna funkcja, 3 tryby)
df_cmp = df_ew[['date', 'Q_up', 'Q_down']].copy()

for method in ['daily_n', 'global_n', 'linear']:
    df_cmp[f'Q_mew_{method}'] = interpolate_q_to_location(
        Q_up=df_ew['Q_up'].values, Q_down=df_ew['Q_down'].values,
        A_up=A_UP, A_down=A_DOWN, A_target=A_TARGET,
        method=method,
    )

# Dla informacji: globalny wykladnik n
n_global = np.log(df_ew['Q_down'].mean() / df_ew['Q_up'].mean()) / np.log(A_DOWN / A_UP)

print(f'Globalny wykladnik n = {n_global:.4f}')
print()
print('Statystyki trzech metod [m3/s]:')
print(df_cmp[['Q_mew_daily_n', 'Q_mew_global_n', 'Q_mew_linear']].describe().round(2))
print()
print('Korelacja miedzy metodami:')
print(df_cmp[['Q_mew_daily_n', 'Q_mew_global_n', 'Q_mew_linear']].corr().round(4))

# Wykres roznic dla ostatniego roku
last_y = df_cmp[df_cmp['date'] >= df_cmp['date'].max() - pd.Timedelta(days=365)]
fig = go.Figure()
fig.add_trace(go.Scatter(x=last_y['date'], y=last_y['Q_mew_daily_n'],
    mode='lines', name='(a) per-day n', line=dict(color='gray', width=1)))
fig.add_trace(go.Scatter(x=last_y['date'], y=last_y['Q_mew_global_n'],
    mode='lines', name=f'(b) global n={n_global:.3f}', line=dict(color='royalblue', width=2)))
fig.add_trace(go.Scatter(x=last_y['date'], y=last_y['Q_mew_linear'],
    mode='lines', name='(c) liniowa po polu', line=dict(color='firebrick', width=2, dash='dot')))
fig.update_layout(
    title='Trzy metody interpolacji Q_ew — ostatni rok',
    xaxis_title='Data', yaxis_title='Q [m3/s]',
    height=400, hovermode='x unified',
)
fig.show()

print()
print('Dla dalszych obliczen uzywamy metody (a) — taka sama jak w arkuszu WPE_2.xlsm.')
print('Jesli zlewnie sa **silnie rozne** (np. dolny dwa razy wiekszy niz gorny), metoda (a)')
print('jest matematycznie najbardziej elastyczna. Dla naszego przypadku roznice sa minimalne.')

Globalny wykladnik n = 3.3596

Statystyki trzech metod [m3/s]:
       Q_mew_daily_n  Q_mew_global_n  Q_mew_linear
count       14975.00        14975.00      14975.00
mean          130.77          132.01        132.03
std            88.92           91.21         89.37
min             9.64            7.97         11.38
25%            68.02           66.01         68.67
50%           109.32          109.27        110.00
75%           174.67          178.00        175.19
max           812.07          769.14        822.24

Korelacja miedzy metodami:
                Q_mew_daily_n  Q_mew_global_n  Q_mew_linear
Q_mew_daily_n          1.0000           0.995        0.9995
Q_mew_global_n         0.9950           1.000        0.9970
Q_mew_linear           0.9995           0.997        1.0000



Dla dalszych obliczen uzywamy metody (a) — taka sama jak w arkuszu WPE_2.xlsm.
Jesli zlewnie sa **silnie rozne** (np. dolny dwa razy wiekszy niz gorny), metoda (a)
jest matematycznie najbardziej elastyczna. Dla naszego przypadku roznice sa minimalne.


### Uwaga numeryczna: czy wykladnik $n$ liczony na dzien ma sens?

Statystyki powyzej pokazuja `n` ze srednia $\approx -0.78$ ale odchyleniem standardowym $\approx 10$.
To **alarmujace** — odchylenie 13× wiekszy niz wartosc srednia oznacza, ze wykladnik dnia po dniu
jest zdominowany przez szum, nie przez fizyke. Przyczyna:

- mianownik: $\ln(A_{down}/A_{up}) = \ln(26860/26495) = \ln(1.0138) \approx 0.0137$ — bardzo maly,
- licznik $\ln(Q_{down}/Q_{up})$ — szumi nawet o $\pm 0.05$ z dnia na dzien,
- dzielenie szumu przez maly mianownik daje **dziki wykladnik**.

Wykladnik $n$ ma sens dopiero **statystycznie** (sredni dla wielolecia), bo wtedy szum sie usrednia.
Trzy poprawne metody do porownania:

1. **Pelna formula z dziennym $n(t)$** (powyzej) — niestabilna numerycznie dla zblizonych zlewni,
2. **Pojedynczy roczny $n$**: $n = \ln(\bar{Q}_{down}/\bar{Q}_{up}) / \ln(A_{down}/A_{up})$, ten sam wykladnik kazdego dnia,
3. **Interpolacja liniowa** wedlug pola: $Q_{ew} = Q_{up} + (Q_{down} - Q_{up}) \cdot (A_{ew} - A_{up})/(A_{down} - A_{up})$ — bez logarytmow, intuicyjna.

**Prompt do LLM:**
> *"Porownaj trzy metody interpolacji Q_ew: (a) per-day n, (b) jeden globalny n,
> (c) liniowa interpolacja po polu. Wyswietl statystyki i wykres roznic dla ostatniego roku."*

**Uzyte funkcje:** `numpy` (logarytmy, srednie)

**Prompt do LLM:**
> *"Narysuj wykres porownawczy przeplywow z trzech lokalizacji (Malczyce, Brzeg Dolny,
> EW interpolowany) na osi czasu -- ostatnie 2 lata. Uzyj plotly."*

In [5]:
# Porownanie przeplywow -- ostatnie 2 lata
# Uwaga: STATION_UP = Brzeg Dolny (Q_up), STATION_DOWN = Malczyce (Q_down)
last_2y = df_ew[df_ew['date'] >= df_ew['date'].max() - pd.Timedelta(days=730)]

fig = go.Figure()
fig.add_trace(go.Scatter(x=last_2y['date'], y=last_2y['Q_up'],
    mode='lines', name=f'Brzeg Dolny — upstream (A={A_UP} km2)', line=dict(width=1)))
fig.add_trace(go.Scatter(x=last_2y['date'], y=last_2y['Q_down'],
    mode='lines', name=f'Malczyce — downstream (A={A_DOWN} km2)', line=dict(width=1)))
fig.add_trace(go.Scatter(x=last_2y['date'], y=last_2y['Q_ew'],
    mode='lines', name=f'EW interpolowany (A={A_TARGET} km2)', line=dict(color='red', width=2)))
fig.update_layout(
    title='Porownanie przeplywow -- Brzeg Dolny, Malczyce, lokalizacja EW',
    xaxis_title='Data', yaxis_title='Q [m3/s]', height=450, hovermode='x unified',
)
fig.show()

---
## Krok 2: Sredni rok uporzadkowany dla lokalizacji EW

Budujemy sredni rok uporzadkowany z interpolowanych przeplywow Q_ew.

**Prompt do LLM:**
> *"Napisz kod ktory: (1) utworzy DataFrame z kolumnami station_id, date, discharge_m3s
> z interpolowanych przeplywow Q_ew (station_id = 'EW'), (2) obliczy sredni rok uporzadkowany
> (lata juz przefiltrowane w nb 02), (3) narysuje wykres FDC (srednia + zakres min-max)
> dla lokalizacji EW."*

**Uzyte moduly/funkcje:** `src.hydrology.average_sorted_year()`.

In [6]:
# Utworz DataFrame w formacie zgodnym z modulem hydrology
# Dane wejsciowe sa juz przefiltrowane w nb 02 — bez lat niekompletnych/powodziowych.
df_mew_hydro = pd.DataFrame({
    'station_id': 'EW',
    'date': df_ew['date'],
    'discharge_m3s': df_ew['Q_ew'],
})

avg_year = average_sorted_year(df_mew_hydro, 'EW')

print(f'Sredni rok uporzadkowany z {avg_year["n_years"].iloc[0]} lat')
print(f'Q: {avg_year["mean"].min():.2f} -- {avg_year["mean"].max():.2f} m3/s')

# Wykres FDC
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=avg_year['exceedance_pct'], y=avg_year['max'],
    mode='lines', line=dict(width=0), showlegend=False))
fig.add_trace(go.Scatter(
    x=avg_year['exceedance_pct'], y=avg_year['min'],
    mode='lines', line=dict(width=0),
    fill='tonexty', fillcolor='rgba(65,105,225,0.15)', name='Zakres min-max'))
fig.add_trace(go.Scatter(
    x=avg_year['exceedance_pct'], y=avg_year['mean'],
    mode='lines', line=dict(color='royalblue', width=2.5), name='Srednia'))
fig.update_layout(
    title=f'Sredni rok uporzadkowany -- lokalizacja EW (A={A_TARGET} km2)',
    xaxis_title='Prawdopodobienstwo przekroczenia [%]',
    yaxis_title='Q [m3/s]', height=500, hovermode='x unified',
)
fig.show()

Sredni rok uporzadkowany z 41 lat
Q: 37.46 -- 439.89 m3/s


---
## Krok 3: Parametry instalowane

Definiujemy parametry fizyczne i sprawnosci EW:

| Parametr | Oznaczenie | Wartosc |
|----------|-----------|---------|
| Gestosc wody | rho | 998 kg/m3 |
| Przyspieszenie ziemskie | g | 9.81 m/s2 |
| Spad brutto | H_gross | 6 m |
| Sprawnosc turbiny | eta_t | 0.92 |
| Sprawnosc generatora | eta_g | 0.96 |
| Sprawnosc wlotu | eta_inlet | 0.98 |
| Sprawnosc wylotu | eta_outlet | 0.98 |
| Min. wspolczynnik przeplywu | Q_min_coeff | 0.15 (15% Q_design) |
| Min. wspolczynnik spadu | H_min_coeff | 0.50 (50% H_gross) |

**Prompt do LLM:**
> *"Napisz kod definiujacy parametry instalowane EW jako slownik. Oblicz sprawnosc calkowita
> jako iloczyn wszystkich sprawnosci czastkowych. Wyswietl parametry."*

In [7]:
# Parametry instalowane
params = {
    'rho': 998,         # gestosc wody [kg/m3]
    'g': 9.81,          # przyspieszenie ziemskie [m/s2]
    'H_gross': 6.0,     # spad brutto [m]
    'eta_t': 0.92,      # sprawnosc turbiny [-]
    'eta_g': 0.96,      # sprawnosc generatora [-]
    'eta_inlet': 0.98,  # sprawnosc wlotu [-]
    'eta_outlet': 0.98, # sprawnosc wylotu [-]
    'Q_min_coeff': 0.15, # minimalny przeplyw (ulamek Q_design)
    'H_min_coeff': 0.50, # minimalny spad (ulamek H_gross)
}

# Sprawnosc calkowita
params['eta_total'] = (params['eta_t'] * params['eta_g']
                       * params['eta_inlet'] * params['eta_outlet'])

print('Parametry instalowane EW:')
for k, v in params.items():
    print(f'  {k:15s} = {v}')
print(f'\nSprawnosc calkowita: eta_total = {params["eta_total"]:.6f}')

Parametry instalowane EW:
  rho             = 998
  g               = 9.81
  H_gross         = 6.0
  eta_t           = 0.92
  eta_g           = 0.96
  eta_inlet       = 0.98
  eta_outlet      = 0.98
  Q_min_coeff     = 0.15
  H_min_coeff     = 0.5
  eta_total       = 0.84822528

Sprawnosc calkowita: eta_total = 0.848225


---
## Krok 4: Model spadu netto

Spad netto zalezy od stanu wody dolnej -- przy wiekszym przeplywu stan wody rosnie,
co zmniejsza spad uzytkowy.

**Podejscie uproszczone:** nie dysponujemy szczegolowa charakterystyka przekroju
poprzecznego koryta przy stopniu wodnym, dlatego zmiennosc poziomu wody dolnej
przyblizymy na podstawie zmian stanow wody na wodowskazie.

Mozliwe warianty:
- **jeden wodowskaz** (np. dolny -- Malczyce)
- **srednia z dwoch wodowskazow** (gorny i dolny)
- **interpolacja** miedzy wodowskazami (proporcjonalnie do zlewni)

Budujemy **sredni rok uporzadkowany** dla stanow wody (ta sama metoda co dla przeplywow),
a nastepnie:

$$H_{day} = H_{stage} + \overline{level} - level_{day}$$

gdzie:
- $H_{stage}$ -- spad brutto na stopniu [m]
- $\overline{level}$ -- sredni stan wody z wielolecia [m]
- $level_{day}$ -- stan wody w danym dniu sredniego roku uporzadkowanego [m]

Parowanie po ranku: najwyzszy przeplyw (ranga 1) odpowiada najwyzszemu stanowi wody
(najnizszy spad), najnizszy przeplyw (ranga 365) -- najnizszemu stanowi (najwyzszy spad).

**Prompt do LLM:**
> *"Napisz kod ktory: (1) obliczy sredni rok uporzadkowany dla stanow wody na wodowskazie
> dolnym (Malczyce) uzywajac average_sorted_year z column='water_level_cm',
> (2) obliczy sredni stan wody z wielolecia, (3) dla kazdego dnia (rangi) obliczy
> spad netto: H_day = H_stage + level_avg - level_day, (4) narysuje wykresy Q(t) i H(t)
> vs procent przekroczenia obok siebie."*

**Uzyte moduly/funkcje:** `src.hydrology.average_sorted_year(column='water_level_cm')`

In [8]:
# Sredni rok uporzadkowany dla stanow wody (wodowskaz dolny -- Malczyce).
# Dane juz przefiltrowane w nb 02 (te same lata co dla Q).
avg_level = average_sorted_year(df, STATION_DOWN, column='water_level_cm')

# Sredni stan wody z wielolecia [m]
all_levels = df.loc[
    df['station_id'] == STATION_DOWN, 'water_level_cm'
].dropna().values / 100.0
level_avg = all_levels.mean()

# Sredni rok uporzadkowany stanow wody [m]
level_sorted = avg_level['mean'].values / 100.0  # cm -> m

print(f'Sredni rok uporzadkowany stanow wody z {avg_level["n_years"].iloc[0]} lat')
print(f'Sredni stan wody: {level_avg:.3f} m')
print(f'Zakres stanow (sr. rok): {level_sorted.min():.3f} -- {level_sorted.max():.3f} m')

# Spad netto: H_day = H_stage + level_avg - level_day
Q_sorted = avg_year['mean'].values
H_net = params['H_gross'] + (level_avg - level_sorted)
H_net = np.maximum(H_net, 0)

print(f'\nSpad netto: {H_net.min():.3f} -- {H_net.max():.3f} m')
print(f'Spad przy srednim stanie wody: {params["H_gross"]:.3f} m (= H_stage)')

Sredni rok uporzadkowany stanow wody z 41 lat
Sredni stan wody: 1.621 m
Zakres stanow (sr. rok): 0.410 -- 4.642 m

Spad netto: 2.979 -- 7.212 m
Spad przy srednim stanie wody: 6.000 m (= H_stage)


In [9]:
# Wykres: Q(t) i H(t) vs procent przekroczenia
fig2 = make_subplots(rows=1, cols=2,
    subplot_titles=['Przeplyw Q(t)', 'Spad netto H(t)'],
    horizontal_spacing=0.12)

pct = avg_year['exceedance_pct'].values

fig2.add_trace(go.Scatter(x=pct, y=Q_sorted, mode='lines',
    line=dict(color='royalblue', width=2), name='Q'), row=1, col=1)
fig2.add_trace(go.Scatter(x=pct, y=H_net, mode='lines',
    line=dict(color='firebrick', width=2), name='H'), row=1, col=2)

fig2.update_xaxes(title_text='Prawdopodobienstwo przekroczenia [%]', row=1, col=1)
fig2.update_xaxes(title_text='Prawdopodobienstwo przekroczenia [%]', row=1, col=2)
fig2.update_yaxes(title_text='Q [m3/s]', row=1, col=1)
fig2.update_yaxes(title_text='H [m]', row=1, col=2)
fig2.update_layout(height=400, showlegend=False,
    title_text='Sredni rok uporzadkowany -- przeplyw i spad netto')
fig2.show()

---
## Krok 5: Obliczenie mocy i energii dla wybranego punktu projektowego

### Punkt projektowy (na krzywej uporzadkowanej)

**Prawdopodobienstwo przekroczenia** $p$% oznacza, ze przeplyw $Q(p)$ jest
przekroczony (lub rowny) przez $p$% czasu w roku.

Punkt projektowy wyznacza **przeplyw projektowy** $Q_{design} = Q(p)$
oraz **spad projektowy** $H_{design} = H_{net}(p)$ (spad w tym samym dniu sredniego roku).

### Przepustowosc turbiny — spad jako kwadrat przeplywu

Cala droga wodna razem z turbina (rurociag, krata, kolana, spirala, wirnik, rura ssawna)
zachowuje sie jak **opor hydrauliczny o stalej charakterystyce** — spad pobrany przez instalacje
jest proporcjonalny do **kwadratu przeplywu**:

$$\Delta H = R_{inst} \cdot Q^2$$

(ta sama forma kwadratowa, co pojedyncze straty miejscowe z nb 06: $\xi \cdot v^2/(2g) = \xi/(2gA^2) \cdot Q^2$).

Stala $R_{inst}$ [s²/m⁵] wyznaczamy z punktu projektowego — tak, zeby krzywa
przechodzila przez $(Q_{design},\, H_{design})$:

$$R_{inst} = \dfrac{H_{design}}{Q_{design}^2}$$

Stad dla dowolnego dnia z dostepnym spadem $H_{net}(t)$, turbina przepuszcza:

$$Q_t(t) = \sqrt{\dfrac{H_{net}(t)}{R_{inst}}}$$

(co jest matematycznie rownowazne $Q_{design}\sqrt{H_{net}(t)/H_{design}}$).

Co to znaczy w praktyce:

- $H = H_{design}$: $Q_t = Q_{design}$ — punkt projektowy,
- $H > H_{design}$ (niskie przeplywy, wyzszy spad): $Q_t > Q_{design}$, ale natywne $Q$ i tak nas ogranicza,
- $H < H_{design}$ (wysokie przeplywy, niski spad): $Q_t < Q_{design}$, czesc wody idzie na przelew.

Rzeczywisty przeplyw przez turbine to **mniejsza** z dwoch wielkosci:

$$
Q_{wpp}(t) =
\begin{cases}
0 & \text{jesli } Q(t) < Q_{min} \;\text{lub}\; H(t) < H_{min}, \\
\min\big(Q(t),\; Q_t(t)\big) & \text{w przeciwnym razie}.
\end{cases}
$$

gdzie $Q_{min} = c_{Q,min} \cdot Q_{design}$ i $H_{min} = c_{H,min} \cdot H_{design}$.

**Odniesienie:** identyczna formula w arkuszu WPE_2.xlsm: `E16 = C16/B16²` definiuje $R_{inst}$,
`D32 = sqrt(C32/E16)` liczy $Q_t$ kazdego dnia.

### Moc i energia

$$P(t) = \rho \cdot g \cdot Q_{wpp}(t) \cdot H_{net}(t) \cdot \eta_{total} \quad [\mathrm{W}]$$

$$E_{roczna} = \sum_{t=1}^{365} P(t) \cdot 24\,\mathrm{h} \quad [\mathrm{kWh/rok}]$$

Mniejszy procent przekroczenia = **wieksza** turbina = wieksza moc ale mniej dni pracy.
Wiekszy procent = **mniejsza** turbina = mniejsza moc ale wiecej dni pracy.
Optymalne $p$ jest gdzies **pomiedzy** — wyznaczymy je w Kroku 6.

**Prompt do LLM:**
> *"Napisz funkcje power_calculation(Q_sorted, H_net, rho, g, eta_total, install_day,
> Q_min_coeff, H_min_coeff) ktora: (1) wyznacza Q_design z Q_sorted[install_day-1],
> (2) oblicza Qmin i Hmin, (3) dla kazdego dnia oblicza Qwpp = min(Q, Q_design)
> jesli Q >= Qmin i H >= Hmin, inaczej 0, (4) oblicza moc P = rho*g*Qwpp*H*eta
> i energie E = P*24 [kWh], (5) zwraca DataFrame z kolumnami: day, Q, H, Q_design,
> Qwpp, P_kW, E_kWh. Nastepnie uzyj tej funkcji dla dnia instalacyjnego = 70
> i narysuj wykresy P(t) i skumulowanej E(t)."*

In [10]:
def power_calculation(Q_sorted, H_net, rho, g, eta_total,
                      install_day, Q_min_coeff=0.15, H_min_coeff=0.5):
    """Oblicz moc i energie EW dla danego dnia instalacyjnego.

    Model hydrauliczny instalacji:
        dH = R_inst * Q^2,    gdzie R_inst = H_design / Q_design^2
        Q_t(H) = sqrt(H / R_inst)             [przepustowosc turbiny przy aktualnym spadzie]
    Przy niskim spadzie turbina przepuszcza mniej niz Q_design.

    Args:
        Q_sorted: posortowane przeplywy (365 wartosci, malejaco)
        H_net: spad netto dla kazdego dnia (365 wartosci)
        rho: gestosc wody [kg/m3]
        g: przyspieszenie ziemskie [m/s2]
        eta_total: sprawnosc calkowita [-]
        install_day: dzien instalacyjny (1-365)
        Q_min_coeff: minimalny przeplyw jako ulamek Q_design
        H_min_coeff: minimalny spad jako ulamek H_design

    Returns:
        DataFrame z kolumnami: day, pct, Q, H, Q_design, R_inst, Q_t, Qwpp, P_kW, E_kWh
    """
    assert 1 <= install_day <= len(Q_sorted), f'install_day={install_day} poza zakresem'
    assert len(Q_sorted) == len(H_net), 'Q_sorted i H_net musza miec te sama dlugosc'

    n_days = len(Q_sorted)
    Q_design = Q_sorted[install_day - 1]
    H_design = H_net[install_day - 1]
    Q_min = Q_min_coeff * Q_design
    H_min = H_min_coeff * H_design

    # Stala hydrauliczna calej instalacji (dH = R * Q^2),
    # wyznaczona tak, zeby krzywa przechodzila przez punkt projektowy.
    R_inst = H_design / Q_design ** 2

    # Przepustowosc turbiny w aktualnym dniu: Q_t = sqrt(H/R)
    Q_t = np.sqrt(np.maximum(H_net, 0) / R_inst)

    # Rzeczywisty przeplyw: mniejszy z natywnego Q i przepustowosci Q_t,
    # z ograniczeniami eksploatacyjnymi Q_min, H_min.
    mask = (Q_sorted >= Q_min) & (H_net >= H_min)
    Qwpp = np.where(mask, np.minimum(Q_sorted, Q_t), 0.0)

    P_kW = rho * g * Qwpp * H_net * eta_total / 1000  # W -> kW
    E_kWh = P_kW * 24

    return pd.DataFrame({
        'day': np.arange(1, n_days + 1),
        'pct': np.round(np.arange(1, n_days + 1) / n_days * 100, 2),
        'Q': Q_sorted,
        'H': H_net,
        'Q_design': Q_design,
        'R_inst': R_inst,
        'Q_t': Q_t,
        'Qwpp': Qwpp,
        'P_kW': P_kW,
        'E_kWh': E_kWh,
    })

# Obliczenie dla dnia instalacyjnego = 70
INSTALL_DAY = 70
result = power_calculation(
    Q_sorted, H_net,
    params['rho'], params['g'], params['eta_total'],
    INSTALL_DAY, params['Q_min_coeff'], params['H_min_coeff'],
)

Q_design = result['Q_design'].iloc[0]
H_design = result['H'].iloc[INSTALL_DAY - 1]
R_inst = result['R_inst'].iloc[0]
E_total_MWh = result['E_kWh'].sum() / 1000
P_max = result['P_kW'].max()
days_operating = (result['Qwpp'] > 0).sum()
install_pct = round(INSTALL_DAY / 365 * 100, 1)

print(f'Punkt projektowy: dzien {INSTALL_DAY} ({install_pct}% przekroczenia)')
print(f'  Q_design = {Q_design:.2f} m3/s')
print(f'  H_design = {H_design:.2f} m')
print(f'  R_inst   = {R_inst:.4f} s2/m5    (dH = R * Q^2)')
print(f'  Qmin = {params["Q_min_coeff"] * Q_design:.2f} m3/s  '
      f'Hmin = {params["H_min_coeff"] * H_design:.2f} m')
print(f'  P_max = {P_max:.1f} kW')
print(f'  E_roczna = {E_total_MWh:.1f} MWh/rok')
print(f'  Dni pracy: {days_operating} / 365')

Punkt projektowy: dzien 70 (19.2% przekroczenia)
  Q_design = 180.26 m3/s
  H_design = 5.36 m
  R_inst   = 0.0002 s2/m5    (dH = R * Q^2)
  Qmin = 27.04 m3/s  Hmin = 2.68 m
  P_max = 8027.7 kW
  E_roczna = 48166.6 MWh/rok
  Dni pracy: 365 / 365


**Prompt do LLM:**
> *"Narysuj dwa wykresy: (1) moc EW P(t) vs dzien roku z zaznaczonym Q_design i Qmin,
> (2) skumulowana energia E(t). Na wykresie mocy zaznacz strefami: pelna moc (Q > Q_design),
> czesciowa moc, i brak pracy (Q < Qmin)."*

In [11]:
# Wykresy: moc i skumulowana energia vs procent przekroczenia
fig = make_subplots(rows=2, cols=1,
    subplot_titles=[
        f'Moc -- punkt projektowy {install_pct}% (Q_design = {Q_design:.1f} m3/s)',
        f'Skumulowana energia roczna (E = {E_total_MWh:.0f} MWh/rok)',
    ],
    vertical_spacing=0.15)

# Moc -- kolorowana wedlug stref pracy
colors = []
Q_min = params['Q_min_coeff'] * Q_design
for i in range(len(result)):
    if result['Qwpp'].iloc[i] == 0:
        colors.append('lightgray')          # brak pracy (Q < Qmin lub H < Hmin)
    elif result['Q'].iloc[i] >= Q_design:
        colors.append('green')              # pelna moc (Q >= Q_design)
    else:
        colors.append('royalblue')          # czesciowa moc

fig.add_trace(go.Bar(
    x=result['pct'], y=result['P_kW'],
    marker_color=colors, name='Moc [kW]', showlegend=False,
), row=1, col=1)
fig.add_hline(y=P_max, line_dash='dash', line_color='green',
    annotation_text=f'P_max = {P_max:.0f} kW', row=1, col=1)

# Skumulowana energia (MWh)
E_cumsum = result['E_kWh'].cumsum() / 1000
fig.add_trace(go.Scatter(
    x=result['pct'], y=E_cumsum,
    mode='lines', line=dict(color='firebrick', width=2),
    name='E skumulowana', showlegend=False,
), row=2, col=1)

fig.update_xaxes(title_text='Prawdopodobienstwo przekroczenia [%]', row=2, col=1)
fig.update_yaxes(title_text='P [kW]', row=1, col=1)
fig.update_yaxes(title_text='E [MWh]', row=2, col=1)
fig.update_layout(height=700, hovermode='x unified')
fig.show()

---
## Krok 6: Optymalizacja punktu projektowego

Iterujemy po roznych punktach na krzywej uporzadkowanej i dla kazdego obliczamy:
- przeplyw projektowy Q_design
- roczna energie E [MWh/rok]
- srednia moc P_avg [kW]

**Prawdopodobienstwo przekroczenia** p% oznacza, ze przeplyw jest przekroczony
przez p% czasu w roku. Np. punkt 20% = przeplyw przekroczony przez 73 dni/rok.

**Prompt do LLM:**
> *"Napisz kod ktory iteruje po dniach instalacyjnych od 10 do 350 (krok 1) i dla kazdego
> oblicza energie roczna uzywajac power_calculation(). Zbierz wyniki do DataFrame
> z kolumnami: install_day, Q_design, H_design, P_max_kW, E_MWh, days_operating.
> Znajdz optymalny dzien (max E). Narysuj wykres E vs dzien instalacyjny."*

In [12]:
# Optymalizacja -- przeglad punktow projektowych
results_opt = []
for day in range(10, 351):
    r = power_calculation(
        Q_sorted, H_net,
        params['rho'], params['g'], params['eta_total'],
        day, params['Q_min_coeff'], params['H_min_coeff'],
    )
    results_opt.append({
        'install_day': day,
        'install_pct': round(day / 365 * 100, 1),
        'Q_design': r['Q_design'].iloc[0],
        'H_design': r['H'].iloc[day - 1],
        'P_max_kW': r['P_kW'].max(),
        'E_MWh': r['E_kWh'].sum() / 1000,
        'days_operating': (r['Qwpp'] > 0).sum(),
    })

df_opt = pd.DataFrame(results_opt)

# Optymalny punkt (max energia)
best = df_opt.loc[df_opt['E_MWh'].idxmax()]
print(f'Optymalny punkt: {best["install_pct"]:.0f}% przekroczenia (dzien {int(best["install_day"])})')
print(f'  Q_design = {best["Q_design"]:.2f} m3/s')
print(f'  P_max    = {best["P_max_kW"]:.1f} kW')
print(f'  E_roczna = {best["E_MWh"]:.1f} MWh/rok')
print(f'  Dni pracy: {int(best["days_operating"])} / 365')

Optymalny punkt: 4% przekroczenia (dzien 13)
  Q_design = 306.50 m3/s
  P_max    = 10441.2 kW
  E_roczna = 51800.4 MWh/rok
  Dni pracy: 358 / 365


In [13]:
# Wykres: energia roczna i moc vs procent przekroczenia
fig = make_subplots(rows=2, cols=1,
    subplot_titles=['Energia roczna vs punkt projektowy',
                    'Moc maksymalna i dni pracy vs punkt projektowy'],
    vertical_spacing=0.15)

fig.add_trace(go.Scatter(
    x=df_opt['install_pct'], y=df_opt['E_MWh'],
    mode='lines', line=dict(color='firebrick', width=2), name='E [MWh/rok]',
), row=1, col=1)
fig.add_vline(x=best['install_pct'], line_dash='dash', line_color='black',
    annotation_text=f'Optimum: {best["install_pct"]:.0f}%', row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_opt['install_pct'], y=df_opt['P_max_kW'],
    mode='lines', line=dict(color='green', width=2), name='P_max [kW]',
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=df_opt['install_pct'], y=df_opt['days_operating'],
    mode='lines', line=dict(color='orange', width=2, dash='dot'), name='Dni pracy',
), row=2, col=1)

fig.update_xaxes(title_text='Prawdopodobienstwo przekroczenia [%]', row=2, col=1)
fig.update_yaxes(title_text='E [MWh/rok]', row=1, col=1)
fig.update_yaxes(title_text='P_max [kW] / Dni pracy', row=2, col=1)
fig.update_layout(height=700, hovermode='x unified')
fig.show()

---
## Krok 7: Analiza ekonomiczna

W **Kroku 6** znalezlismy punkt projektowy maksymalizujacy **energie roczna**.
Ale wieksza energia = wiekszy plant = **wieksza inwestycja**. Trzeba porownac
zwrot kapitalu w cyklu zycia projektu.

### Pelny model kosztow (Ogayar + IRENA)

Zamiast plaskiego 1300 EUR/kW (poprzednia wersja tego notebooka — uproszczone),
uzywamy modelu skladowego z `src/costs.py` (taki sam jak w nb 09 i nb 10):

| Skladnik | Zrodlo | Wzor / wartosc |
|----------|--------|-----------------|
| Urzadzenia EM | Ogayar & Vidal (2009) × 1.37 inflacji 2009→2024 | $C_{EM} = a \cdot P^b \cdot H^c$ |
| Roboty budowlane | IRENA 2012/2023 | mnoznik × $C_{EM}$ (1.0× dla 'run-of-river small') |
| Przylacze sieciowe | NREL ATB 2024 | 100 EUR/kW |
| Inzynieria | IRENA 2012 | 10% sumy powyzszych |

Plus `economic_analysis()`: O&M 2.5%/rok (IRENA), stopa dyskontowa 6%, czas zycia 40 lat.

### Trzy kryteria optymalizacji daja **rozne odpowiedzi**

| Kryterium | Co maksymalizuje? | Tendencja |
|-----------|-------------------|-----------|
| **Max E** (Krok 6) | Suma kWh wyprodukowanych w roku | Bardzo duzy plant (niskie %) |
| **Min payback** | Szybkosc zwrotu inwestycji | Bardzo maly plant (wysokie %) |
| **Min LCOE** | Najnizszy koszt 1 kWh energii | Sredni plant (umiarkowane %) |
| **Max NPV** | Calkowity zysk w cyklu zycia | Optymalny inwestycyjnie |

W praktyce inwestycyjnej **maksymalizujemy NPV** — to mierzy *calkowity zysk*
z uwzglednieniem czasowej wartosci pieniadza. Min payback to kryterium
**konserwatywne** (faworyzuje male projekty); min LCOE to **dobre porownanie**
z innymi zrodlami energii.

### Uwaga o ograniczeniach

Wzor Ogayara jest walidowany dla **P < 2 MW** — powyzej tej granicy formula
*znacznie zanizyje* koszty (Ogayar ma silna ekonomie skali, ktora nie utrzymuje sie
dla duzych instalacji). Jezeli optimum NPV wypadnie przy mocy > 2 MW, traktuj wynik
jako orientacyjny i porownaj z **pelnym modelem produkcji w nb 10** (zmiennie sprawnosci
+ przeplyw nienaruszalny Q_env).

Dodatkowo ten notebook ustawia udzial robot budowlanych jak dla **malej EW** (`run_of_river_small`, 1.0×); dla optimow > 5 MW koszt robot budowlanych jest rowniez zanizony — nb 09/10 uzywaja `plant_type='auto'`, ktory powyzej 5 MW przelacza na 2.0×.

**Prompt do LLM:**
> *"Dla kazdego dnia instalacyjnego z `df_opt` wywolaj `total_investment` z `src/costs`
> (Kaplan, run-of-river small) i `economic_analysis` (cena 106.5 EUR/MWh, O&M 2.5%,
> dyskonto 6%, lifetime 40 lat). Zbierz NPV, LCOE, payback. Znajdz optima dla trzech
> kryteriow i opisz roznice."*

**Uzyte funkcje:** `src.costs.total_investment`, `src.costs.economic_analysis`, `src.costs.EUR_PLN_RATE`.

In [14]:
import warnings
from src.costs import total_investment, economic_analysis, EUR_PLN_RATE

# Parametry projektu (te same co w nb 09 i nb 10)
ENERGY_PRICE_EUR = 106.5     # 490 PLN/MWh / 4.6
OM_FRACTION = 0.025          # 2.5% rocznie (IRENA 2012)
DISCOUNT_RATE = 0.06         # 6%
LIFETIME = 40                # lat
PLANT_TYPE = 'run_of_river_small'
TURBINE_TYPE_KEY = 'kaplan'
N_TURBINES_SIMPLE = 1        # nb 05 uproszczone: 1 duza turbina (nb 10 ma 2)

# Petla po dniach instalacyjnych — pelny model kosztow + NPV/LCOE/payback
econ_rows = []
for _, row in df_opt.iterrows():
    P_per_unit = row['P_max_kW'] / N_TURBINES_SIMPLE
    H_d = row['H_design']
    if P_per_unit <= 0 or H_d <= 0:
        continue
    # Ostrzezenie Ogayar P>2MW jest spodziewane przy malych dniach instalacyjnych — wyciszamy
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        inv = total_investment(
            P_kW=P_per_unit, H=H_d,
            n_turbines=N_TURBINES_SIMPLE,
            turbine_type=TURBINE_TYPE_KEY,
            plant_type=PLANT_TYPE,
        )
        econ = economic_analysis(
            energy_mwh=row['E_MWh'],
            investment_eur=inv['total_eur'],
            energy_price_eur_mwh=ENERGY_PRICE_EUR,
            om_fraction=OM_FRACTION,
            discount_rate=DISCOUNT_RATE,
            lifetime_years=LIFETIME,
        )
    econ_rows.append({
        'install_day': row['install_day'],
        'install_pct': row['install_pct'],
        'P_max_kW': row['P_max_kW'],
        'E_MWh': row['E_MWh'],
        'investment_eur': inv['total_eur'],
        'investment_per_kw_eur': inv['total_per_kw_eur'],
        'em_eur': inv['em_eur'],
        'civil_eur': inv['civil_eur'],
        'npv_eur': econ['npv_eur'],
        'lcoe_eur_mwh': econ['lcoe_eur_mwh'],
        'payback_years': econ['payback_years'],
        'over_2mw': P_per_unit > 2000,
    })

df_econ = pd.DataFrame(econ_rows)

# Trzy optymalne punkty
best_npv = df_econ.loc[df_econ['npv_eur'].idxmax()]
best_lcoe = df_econ.loc[df_econ['lcoe_eur_mwh'].idxmin()]
best_payback = df_econ.loc[df_econ['payback_years'].idxmin()]

print('=== Optymalizacja ekonomiczna (Ogayar + IRENA) ===\n')
for label, b in [('Max NPV', best_npv), ('Min LCOE', best_lcoe), ('Min payback', best_payback)]:
    print(f'{label}: dzien {int(b["install_day"])} ({b["install_pct"]:.0f}% przekroczenia)')
    flag = ' ⚠ Ogayar poza zakresem (P > 2 MW)' if b['over_2mw'] else ''
    print(f'  P_max          = {b["P_max_kW"]:>8.0f} kW{flag}')
    print(f'  E_roczna       = {b["E_MWh"]:>8.0f} MWh/rok')
    print(f'  Inwestycja     = {b["investment_eur"]/1e6:>8.2f} mln EUR '
          f'({b["investment_per_kw_eur"]:.0f} EUR/kW)')
    print(f'  NPV            = {b["npv_eur"]/1e6:>+8.2f} mln EUR '
          f'(= {b["npv_eur"]*EUR_PLN_RATE/1e6:+.2f} mln PLN)')
    print(f'  LCOE           = {b["lcoe_eur_mwh"]:>8.1f} EUR/MWh '
          f'(= {b["lcoe_eur_mwh"]*EUR_PLN_RATE:.0f} PLN/MWh)')
    print(f'  Payback        = {b["payback_years"]:>8.1f} lat')
    print()

=== Optymalizacja ekonomiczna (Ogayar + IRENA) ===

Max NPV: dzien 17 (5% przekroczenia)
  P_max          =    10156 kW ⚠ Ogayar poza zakresem (P > 2 MW)
  E_roczna       =    51744 MWh/rok
  Inwestycja     =     5.21 mln EUR (513 EUR/kW)
  NPV            =   +75.75 mln EUR (= +348.45 mln PLN)
  LCOE           =      9.2 EUR/MWh (= 42 PLN/MWh)
  Payback        =      1.0 lat

Min LCOE: dzien 111 (30% przekroczenia)
  P_max          =     6970 kW ⚠ Ogayar poza zakresem (P > 2 MW)
  E_roczna       =    45002 MWh/rok
  Inwestycja     =     4.15 mln EUR (595 EUR/kW)
  NPV            =   +66.40 mln EUR (= +305.46 mln PLN)
  LCOE           =      8.4 EUR/MWh (= 39 PLN/MWh)
  Payback        =      0.9 lat

Min payback: dzien 111 (30% przekroczenia)
  P_max          =     6970 kW ⚠ Ogayar poza zakresem (P > 2 MW)
  E_roczna       =    45002 MWh/rok
  Inwestycja     =     4.15 mln EUR (595 EUR/kW)
  NPV            =   +66.40 mln EUR (= +305.46 mln PLN)
  LCOE           =      8.4 EUR/MWh (= 39 

In [15]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
    subplot_titles=['NPV [mln EUR]', 'LCOE [EUR/MWh]', 'Payback [lat]'],
    vertical_spacing=0.08)

x = df_econ['install_pct']

# NPV
fig.add_trace(go.Scatter(x=x, y=df_econ['npv_eur']/1e6,
    mode='lines', line=dict(color='royalblue', width=2.5), name='NPV'),
    row=1, col=1)
# Zaznacz fragment poza zakresem Ogayar
mask_out = df_econ['over_2mw']
if mask_out.any():
    fig.add_trace(go.Scatter(x=x[mask_out], y=(df_econ['npv_eur']/1e6)[mask_out],
        mode='lines', line=dict(color='orange', width=4),
        name='P > 2 MW (Ogayar poza zakresem)'), row=1, col=1)
fig.add_vline(x=best_npv['install_pct'], line=dict(color='royalblue', dash='dash'),
    annotation_text=f'opt NPV: {best_npv["install_pct"]:.0f}%', row=1, col=1)
fig.add_hline(y=0, line=dict(color='gray', width=0.5), row=1, col=1)

# LCOE
fig.add_trace(go.Scatter(x=x, y=df_econ['lcoe_eur_mwh'],
    mode='lines', line=dict(color='green', width=2.5), name='LCOE'),
    row=2, col=1)
fig.add_vline(x=best_lcoe['install_pct'], line=dict(color='green', dash='dash'),
    annotation_text=f'opt LCOE: {best_lcoe["install_pct"]:.0f}%', row=2, col=1)
fig.add_hline(y=ENERGY_PRICE_EUR, line=dict(color='gray', dash='dot'),
    annotation_text=f'Cena energii = {ENERGY_PRICE_EUR} EUR/MWh', row=2, col=1)

# Payback
fig.add_trace(go.Scatter(x=x, y=df_econ['payback_years'].clip(0, 50),
    mode='lines', line=dict(color='firebrick', width=2.5), name='Payback'),
    row=3, col=1)
fig.add_vline(x=best_payback['install_pct'], line=dict(color='firebrick', dash='dash'),
    annotation_text=f'opt payback: {best_payback["install_pct"]:.0f}%', row=3, col=1)

fig.update_xaxes(title_text='Prawdopodobienstwo przekroczenia [%]', row=3, col=1)
fig.update_yaxes(title_text='NPV [mln EUR]', row=1, col=1)
fig.update_yaxes(title_text='LCOE [EUR/MWh]', row=2, col=1)
fig.update_yaxes(title_text='Lata', row=3, col=1)
fig.update_layout(height=850, showlegend=True,
    title=f'Optymalizacja ekonomiczna — Kaplan, {N_TURBINES_SIMPLE} turbina, '
          f'cena {ENERGY_PRICE_EUR} EUR/MWh, dyskonto {DISCOUNT_RATE:.0%}, T={LIFETIME} lat')
fig.show()

### Co mowi wynik?

**Jezeli optima sa zbiezne** (NPV, LCOE, payback wszystkie w okolicy ~30%):
model jest spojny i punkt projektowy jest jednoznaczny.

**Jezeli sie rozchodza** (jak w naszym przypadku) — to **prawdziwa** charakterystyka
modeli ekonomicznych, nie blad:

- **NPV faworyzuje** duze plantu, jezeli model kosztow ma silna ekonomie skali
  (Ogayar ma — koszt na kW maleje znaczaco z $P^{-0.58}$). Czesto NPV optimum
  wypada w obszarze, gdzie Ogayar **wykracza poza swoj zakres walidacji** (P > 2 MW)
  — to znak, ze trzeba uzyc lepszego modelu.
- **Payback faworyzuje** male projekty (szybki zwrot), ale ignoruje cykl zycia.
- **LCOE** zwykle wskazuje na umiarkowany plant — to dobre kryterium projektowe.

### Praktyka inzynierska

Dla **niskospadowych elektrowni na duzych rzekach** (jak nasz przypadek) typowy
dzien instalacyjny wynosi **~80-120** (~22-33% przekroczenia). Daje to balans:
moc zainstalowana wystarczajaca aby wykorzystac szczyt przeplywu, a koszt
nie eksploduje na kosztach robot budowlanych dla zbyt duzego plantu.

**Dla pelnego porownania** zobacz **nb 10**, gdzie model uwzglednia:
- **zmienna sprawnosc turbiny** $\eta_t(Q/Q_{des})$ — przy niskim obciazeniu spada
- **zmienna sprawnosc generatora** $\eta_g(P/P_n)$ — peak ~80% obciazenia
- **przeplyw nienaruszalny** $Q_{env} \approx Q_{90\%}$ rezerwowany dla rzeki
- **realny rozdzielacz wieloturbinowy** (2 turbiny lepiej skaluja niskie przeplywy)

Te efekty wspolnie **karzace** za zbyt duzy plant (czesc roku stoi przy niskim obciazeniu),
co przesowa optimum NPV do realistycznego ~100 dnia. nb 05 (uproszczone, stale $\eta$)
**zawsze** bedzie mial optimum przesuniete w stosunku do nb 10.

---
## Podsumowanie

W tym notebooku:
1. **Interpolacja** przeplywu do lokalizacji EW miedzy Malczyce a Brzeg Dolny (3 metody)
2. **Sredni rok uporzadkowany** dla lokalizacji EW (przeplyw i stan wody)
3. **Model spadu netto** -- H(Q) z uwzglednieniem zmian stanu wody dolnej
4. **Obliczenie mocy i energii** ze stalym $\eta_{total}$ i charakterystyka turbiny $\Delta H = R \cdot Q^2$
5. **Optymalizacja max-E** dnia instalacyjnego (Krok 6)
6. **Analiza ekonomiczna** z modelem **Ogayar + IRENA** i kryteriami NPV / LCOE / payback (Krok 7)

**Wynik:** rozne kryteria daja rozne optymalne dni instalacyjne. NPV i LCOE sa
lepsze niz payback. Realistyczne wartosci uzyskasz w **nb 10**.

**Dalej:** nb 06 (straty hydrauliczne) → nb 07 (turbina) → nb 08 (generator)
→ nb 09 (koszty) → **nb 10 (integracja, pelen model produkcji)**